# Crypto Portfolio Tracker with Real-Time Calculations


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto Portfolio Tracker with Real-Time Calculations

This notebook builds a functional crypto portfolio tracker inspired by Excel templates, computing key metrics: holdings, cost basis, current values, P&L, and allocation percentages. It uses publicly available data sources and can be adapted to track any portfolio.

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime

# Sample portfolio data
portfolio_data = {
    'Coin': ['Bitcoin', 'Ethereum', 'Cardano'],
    'Quantity': [0.5, 5.0, 500.0],
    'Purchase_Price_USD': [45000, 3000, 0.80],
    'Purchase_Date': ['2023-01-15', '2023-06-01', '2023-03-20']
}

portfolio_df = pd.DataFrame(portfolio_data)
print("Portfolio Data:")
print(portfolio_df)

In [ ]:
# Fetch current prices from CoinGecko (free, no API key required)
def get_current_prices(coin_ids):
    url = 'https://api.coingecko.com/api/v3/simple/price'
    params = {
        'ids': ','.join(coin_ids),
        'vs_currencies': 'usd',
        'include_market_cap': 'true',
        'include_24hr_vol': 'true'
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching prices: {e}")
        return None

# Map coin names to CoinGecko IDs
coin_mapping = {'Bitcoin': 'bitcoin', 'Ethereum': 'ethereum', 'Cardano': 'cardano'}
coin_ids = [coin_mapping[coin] for coin in portfolio_df['Coin']]

prices = get_current_prices(coin_ids)
if prices:
    current_prices = {coin: prices[coin_mapping[coin]]['usd'] for coin in portfolio_df['Coin']}
    portfolio_df['Current_Price_USD'] = portfolio_df['Coin'].map(current_prices)
    print("\nCurrent Prices (USD):")
    print(portfolio_df[['Coin', 'Current_Price_USD']])
else:
    print("Failed to fetch prices. Using sample prices.")
    portfolio_df['Current_Price_USD'] = [52000, 3500, 1.10]

In [ ]:
# Calculate portfolio metrics
portfolio_df['Cost_Basis'] = portfolio_df['Quantity'] * portfolio_df['Purchase_Price_USD']
portfolio_df['Current_Value'] = portfolio_df['Quantity'] * portfolio_df['Current_Price_USD']
portfolio_df['Unrealized_PL'] = portfolio_df['Current_Value'] - portfolio_df['Cost_Basis']
portfolio_df['ROI_Percent'] = (portfolio_df['Unrealized_PL'] / portfolio_df['Cost_Basis'] * 100).round(2)

print("\nDetailed Portfolio Metrics:")
print(portfolio_df[['Coin', 'Quantity', 'Cost_Basis', 'Current_Value', 'Unrealized_PL', 'ROI_Percent']].to_string())

In [ ]:
# Portfolio allocation and summary
total_cost_basis = portfolio_df['Cost_Basis'].sum()
total_current_value = portfolio_df['Current_Value'].sum()
total_unrealized_pl = portfolio_df['Unrealized_PL'].sum()
overall_roi = (total_unrealized_pl / total_cost_basis * 100) if total_cost_basis > 0 else 0

portfolio_df['Allocation_Percent'] = (portfolio_df['Current_Value'] / total_current_value * 100).round(2)

print("\nPortfolio Allocation:")
print(portfolio_df[['Coin', 'Current_Value', 'Allocation_Percent']].to_string())

print(f"\n{'='*50}")
print(f"PORTFOLIO SUMMARY")
print(f"{'='*50}")
print(f"Total Cost Basis:        ${total_cost_basis:,.2f}")
print(f"Total Current Value:     ${total_current_value:,.2f}")
print(f"Total Unrealized P&L:    ${total_unrealized_pl:,.2f}")
print(f"Overall ROI:             {overall_roi:.2f}%")
print(f"Timestamp:               {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: allocation
axes[0].pie(portfolio_df['Current_Value'], labels=portfolio_df['Coin'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Portfolio Allocation by Value')

# Bar chart: ROI by coin
colors = ['green' if x > 0 else 'red' for x in portfolio_df['ROI_Percent']]
axes[1].bar(portfolio_df['Coin'], portfolio_df['ROI_Percent'], color=colors, alpha=0.7)
axes[1].set_title('Return on Investment by Coin')
axes[1].set_ylabel('ROI (%)')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Portfolio visualization complete.")

## How to Use This Tracker

1. **Edit portfolio_data**: Modify the initial dictionary with your actual coins, quantities, and purchase prices.
2. **Update Purchase Dates**: Add actual transaction dates for future tax-reporting features.
3. **Run Cells**: Execute sequentially to fetch live prices and calculate metrics.
4. **Export Results**: Use `portfolio_df.to_csv('my_portfolio.csv')` to save locally.

This notebook can be extended with tax-loss harvesting calculations, transaction history, and rebalancing alerts.

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/crypto/crypto-portfolio-tracker-excel-free-download)
